# 11 language-script group dataset for continued fine-tuning

Builds the dataset for the continued fine-tuning experiment across exactly
**11 language-script groups**.

| Group | label (ISO-3) | script | Source |
| --- | --- | --- | --- |
| `Sinhala-Sinh` | `sin` | `Sinh` | existing target `train.csv` / `val.csv` |
| `Pali-Sinh` | `pli` | `Sinh` | existing target `train.csv` / `val.csv` |
| `Sanskrit-Sinh` | `san` | `Sinh` | existing target `train.csv` / `val.csv` |
| `Sanskrit-Deva` | `san` | `Deva` | `surajp/sanskrit_classic` |
| `English-Latn` | `eng` | `Latn` | `CohereLabs/aya_dataset` |
| `Tamil-Taml` | `tam` | `Taml` | `CohereLabs/aya_dataset` |
| `Hindi-Deva` | `hin` | `Deva` | `CohereLabs/aya_dataset` |
| `Bengali-Beng` | `ben` | `Beng` | `CohereLabs/aya_dataset` |
| `Arabic-Arab` | `arb` | `Arab` | `CohereLabs/aya_dataset` |
| `French-Latn` | `fra` | `Latn` | `CohereLabs/aya_dataset` |
| `German-Latn` | `deu` | `Latn` | `CohereLabs/aya_dataset` |

All other Aya languages have been removed from this configuration.

Both Sanskrit groups keep `label = "san"`; `script` and `group` distinguish
them. Labels are **not** converted to fastText output IDs here.

## Outputs (new names -- earlier files are preserved)

| File | Contents |
| --- | --- |
| `datasets/finetuning/train_mixed_11groups.jsonl` | Training records, all 11 groups |
| `datasets/finetuning/val_mixed_11groups.jsonl` | Validation records, all 11 groups |
| `datasets/finetuning/dataset_11groups_report.json` | Seed, sources, exclusions, validation findings |

The older `train_mixed.jsonl` / `val_mixed.jsonl` from the previous
22-language rehearsal experiment are **left untouched**.

## Record schema

Each line carries `text`, `label` (ISO-3), `script`, `group`, `source` and
`provenance`, plus whatever original identifiers the source provided
(`orig_id`, `subcorpus`, `group_id`, `aya_row_index`, `line_index`).

## Key properties

- **Target splits preserved.** Sinhala, Pali and Sanskrit-Sinh keep their
  existing train/val assignment verbatim; those files are never recombined
  and randomly re-split.
- **Reproducible sampling.** Seed 42, `random.sample` over the deduplicated
  eligible pool -- not the first N rows. Cap of 5,000 per language,
  configurable via `MAX_PER_LANG`. Sources with fewer eligible examples keep
  everything available; no row is ever duplicated to reach the quota.
- **Grouped 90/10 split** on the newly sampled sources, keyed on the
  NFC-normalized text so identical texts never straddle train and val.
- **Script checked separately from the label.** Metadata alone is not taken
  as proof; mixed-script and low-purity examples are reported and excluded.
- **Benchmarks stay separate.** FLORES+, CommonLID and WiLI-2018 are read
  only for overlap reporting -- never for training, validation, early
  stopping or model selection, and never modified.


In [ ]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas datasets huggingface_hub')
    print("Setup complete!")


## Sanskrit-Devanagari source

`CohereLabs/aya_dataset` contains **no Sanskrit at all** (verified: `san`
appears in 0 of its 70 language codes), so this group needs an independent
source.

We reuse the source already documented for this class in
`docs/DATA_DICTIONARY_11LANG_HYBRID.md`: **`surajp/sanskrit_classic`**. That
HF repo ships only a legacy loader script, so we fetch its upstream data
archive directly and record the provenance (URL + SHA-256) in the report.

This corpus has no predefined splits, so the same cap and 90/10 grouped
split used for the Aya languages is applied to it.

We do **not** transliterate Sinhala-script Sanskrit and relabel it as
Devanagari, and we do **not** take Sanskrit from FLORES+/CommonLID/WiLI.


In [ ]:
# Fetch the Sanskrit-Devanagari corpus (skipped if already present).
import os, io, zipfile, hashlib, urllib.request

if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

SAN_DEVA_DIR = "datasets/raw_download/sanskrit_classic"
SAN_DEVA_FILE = os.path.join(SAN_DEVA_DIR, "combined.txt")
SAN_DEVA_URL = ("https://github.com/parmarsuraj99/hf_datasets/raw/master/"
                "sanskrit_classic/combined.zip")

if os.path.exists(SAN_DEVA_FILE):
    print(f"Already present: {SAN_DEVA_FILE}")
else:
    os.makedirs(SAN_DEVA_DIR, exist_ok=True)
    print(f"Downloading {SAN_DEVA_URL} ...")
    blob = urllib.request.urlopen(SAN_DEVA_URL, timeout=180).read()
    print("  bytes:", len(blob), "sha256:", hashlib.sha256(blob).hexdigest())
    zipfile.ZipFile(io.BytesIO(blob)).extractall(SAN_DEVA_DIR)
    print(f"Extracted to {SAN_DEVA_FILE}")


## Build

All the logic lives in `build_11groups.py` next to this notebook, so the same
code runs from the notebook, the Makefile and the command line. Sampling
settings (`SEED`, `MAX_PER_LANG`, `VAL_FRACTION`, quality thresholds) are
constants at the top of that module.

The build **fails loudly** if the target `train.csv` / `val.csv` are missing,
rather than silently producing an Aya-only dataset. Run
`setup_finetune_data.ipynb` first.


In [ ]:
# Build the 11-group dataset.
import importlib.util, sys

spec = importlib.util.spec_from_file_location(
    "build_11groups", "scripts/05.finetune_dataset/build_11groups.py"
)
build = importlib.util.module_from_spec(spec)
sys.modules["build_11groups"] = build
spec.loader.exec_module(build)

build.main()


In [ ]:
# Inspect the generated report.
import json

report = json.load(open("datasets/finetuning/dataset_11groups_report.json",
                        encoding="utf-8"))

print("seed:", report["seed"])
print("cap per language:", report["sampling"]["max_per_language"])
print("Sanskrit-Deva:", report["sources"]["sanskrit_deva"]["status"])
print()

v = report["validation"]
print(f"groups observed        : {v['observed_group_count']}/11")
print(f"missing groups         : {v['missing_groups'] or 'none'}")
print(f"empty train groups     : {v['groups_with_empty_train'] or 'none'}")
print(f"empty val groups       : {v['groups_with_empty_val'] or 'none'}")
print(f"empty text in output   : {v['empty_text_in_output']}")
print(f"dups within train/val  : {v['duplicates_within_train']}/{v['duplicates_within_val']}")
print(f"train/val overlap      : {v['overlap_train_val']}")
print(f"label conflicts        : {v['conflicting_label_keys']}")
print(f"held-out overlap total : {v['total_held_out_overlap']}")
print()
print("test/benchmark files actually checked:")
for f in v["test_files_checked"]:
    print(f"  {f}: {v['overlap_checks'][f]}")
print()

print(f"{'Group':<16}{'Train':>10}{'Val':>10}")
print("-" * 36)
for g, c in report["counts"]["per_group"].items():
    print(f"{g:<16}{c['train']:>10}{c['val']:>10}")
print("-" * 36)
print(f"{'TOTAL':<16}{report['counts']['train_total']:>10}"
      f"{report['counts']['val_total']:>10}")


## Notes

- Generated data lives under `data_pipeline/datasets/`, which is git-ignored.
- No model is trained by this notebook.
- To reproduce from a shell:
  `cd data_pipeline && python scripts/05.finetune_dataset/build_11groups.py`
